## Imports and data loading

In [ ]:
import pandas as pd
import requests
import zipfile
import io, os
import matplotlib.pyplot as plt
import numpy as np
import sys
import math

from pandas.core.interchange.dataframe_protocol import DataFrame

In [ ]:
# LOADING DATA

# Archive IDs for individual years
gios_archive_url = "https://powietrze.gios.gov.pl/pjp/archives/downloadFile/"
gios_url_ids = {2014: '302', 2019: '322', 2024: '582'}
gios_pm25_file = {2014: '2014_PM2.5_1g.xlsx', 2019: '2019_PM25_1g.xlsx', 2024: '2024_PM25_1g.xlsx'}

# Function for downloading a given archive
def download_gios_archive(year, gios_id, filename):
    # Download the ZIP archive into memory
    url = f"{gios_archive_url}{gios_id}"
    response = requests.get(url)
    response.raise_for_status()  # if HTTP error, stop

    # Open zip in memory
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        # find the correct PM2.5 file
        if not filename:
            print(f"Error: {filename} not found.")
        else:
            # load file into pandas
            with z.open(filename) as f:
                try:
                    df = pd.read_excel(f, header=None)
                except Exception as e:
                    print(f"Error while loading {year}: {e}")
    return df

df2014 = download_gios_archive(2014, gios_url_ids[2014], gios_pm25_file[2014])
df2019 = download_gios_archive(2019, gios_url_ids[2019], gios_pm25_file[2019])
df2024 = download_gios_archive(2024, gios_url_ids[2024], gios_pm25_file[2024])

In [ ]:
# Downloading metadata
def download_metadata(gios_id, filename):
    url = f"{gios_archive_url}{gios_id}"
    response = requests.get(url)
    response.raise_for_status()

    try:
        df = pd.read_excel(io.BytesIO(response.content), header=None)
    except Exception as e:
        print(f"Error while loading {filename}, {e}")
    return df

metadata = download_metadata('584', 'Metadane oraz kody stacji i stanowisk pomiarowych.xlsx')

## Part 1

In [ ]:
## Definitions of file cleaning functions

# Unified format
def standardize_format(df):
    """
    Sets appropriate rows/columns as index, standardizes their format and the format of values.

    :param df: DataFrame where formatting changes will be applied
    :return: updated DataFrame
    """
    df.columns = df.iloc[0]  # first row "Station code", set as header
    df = df[1:].reset_index(drop=True)  # cut this row from df values

    # Convert all values in the first column to datetime type
    first_col = df.columns[0]
    df[first_col] = pd.to_datetime(df[first_col], errors='coerce')

    # Set the first column as index
    df = df.set_index(first_col)

    # Clean/standardize column names
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        ##.str.replace(' ', '_', regex=False)
    )

    # Convert values to numeric
    df = df.apply(pd.to_numeric, errors='coerce')

    return df


# Update station codes
def update_station_codes(df):
    """
    Replaces outdated station codes with new ones, according to information from the metadata.

    :param df: DataFrame where the replacement will be performed
    :return: updated DataFrame
    """
    # col 4 - old code, col 1 - new code
    # Take only rows where the old code is NOT NaN
    code_map = {old: new for old, new in zip(metadata.iloc[1:, 4], metadata.iloc[1:, 1]) if pd.notna(old)}

    # SANITY CHECK 3: Check that the map is not completely empty
    if not code_map:
        print("Warning: No old codes found to map (all were empty). Code map is empty.")

    df = df.rename(columns=code_map)  # Replace column names according to the station name map

    return df


# Remove unique codes
def remove_unique_stations(df, common_codes):
    """
    Returns a DataFrame without columns marked with a station code that does not appear in all other DataFrames.

    :param df: DataFrame where columns will be removed
    :param common_codes: list of codes that appear in every other DataFrame
    :return: updated DataFrame
    """
    return df[common_codes]


# Create MultiIndex in headers
def add_multiindex_headers(df, combined_headers):
    """
    Creates MultiIndex headers (City, Station code).

    :param df: DataFrame where MultiIndex will be added
    :param combined_headers: list of tuples of station codes and their corresponding cities
    :return: updated DataFrame
    """
    df.columns = pd.MultiIndex.from_tuples(combined_headers, names=['City', 'Station code'])
    return df


# Shift midnight timestamps to the previous day
def shift_midnight_to_previous_day(df):
    """
    Finds dates in the index column where the time equals 00:00:00 and shifts the calendar day back by 1.

    :param df: DataFrame where the changes will be made
    :return: updated DataFrame
    """
    midnight_mask = df.index.time == pd.to_datetime('00:00:00').time()
    # Subtract 1 or 0 days from the index column
    df.index = df.index - pd.to_timedelta(midnight_mask.astype(int), unit='d')
    return df


# Check whether files have an equal number of columns
def check_equal_station_count(dfs):
    """
    Checks whether cleaning succeeded - whether all files have the same number of columns.
    Exits the program if there is a mismatch.

    :param dfs: list of DataFrames to compare
    """
    station_counts = [df.shape[1] for df in dfs]
    are_equal = len(set(station_counts)) == 1
    if not are_equal:
        sys.exit('Error: The number of columns in the files differs')


# Check whether files have the correct number of days in the year
def check_correct_day_count(dfs):
    """
    Checks whether cleaning succeeded - whether each file has the correct number of days (calendar-wise).
    Exits the program if there is a mismatch.

    :param dfs: list of DataFrames to check
    """
    from calendar import isleap
    for df in dfs:
        year = df.index.year[0]
        day_count = df.index.normalize().unique()  # all hours -> 00:00:00, only unique values
        expected_days = 366 if isleap(year) else 365
        if len(day_count) != expected_days:
            sys.exit(f"Error: The number of days in file {year}_PM2.5_1g.xlsx is incorrect")

In [ ]:
# Calling the cleaning functions
def clean_files(dfs):
    """
    Calls the other functions responsible for modifying each considered DataFrame.

    :param dfs: list of DataFrames on which the functions will be executed
    :return: list of appropriately modified DataFrames
    """
    # Standardize format and update station codes
    for i in range(len(dfs)):
        dfs[i] = standardize_format(dfs[i])
        dfs[i] = update_station_codes(dfs[i])

    # Create a set of common station codes
    common_codes = set(dfs[0].columns)
    for df in dfs[1:]:
        common_codes &= set(df.columns)

    # Remove stations not present in all DataFrames
    dfs = [remove_unique_stations(df, list(common_codes)) for df in dfs]

    # Multi-indexing (Station code | City)
    # col 1 - "Station code", col 11 - "City"
    station_city = dict(zip(metadata.iloc[:, 1], metadata.iloc[:, 11]))
    multi_index = [(station_city[code], code) for code in dfs[0].columns]
    dfs = [add_multiindex_headers(df, multi_index) for df in dfs]

    # Shift midnight timestamps to the previous day
    dfs = [shift_midnight_to_previous_day(df) for df in dfs]

    # Run sanity checks
    check_equal_station_count(dfs)
    check_correct_day_count(dfs)

    return dfs

In [ ]:
# Merge DataFrames and prepare for Excel export
def merge_dataframes(dfs):
    """
    Merges the previously prepared DataFrames and creates an xlsx file from them.

    :param dfs: list of DataFrames to merge
    :return: merged DataFrame
    """
    # Concatenate files by rows
    merged_dfs = pd.concat(dfs, axis=0)

    # MultiIndex was flattened => join back
    merged_dfs.columns = [f'{city}_{station}' for city, station in merged_dfs.columns]
    merged_dfs.columns.name = None

    # Reset the datetime index so it is not lost when saving
    merged_dfs = merged_dfs.reset_index()

    return merged_dfs

In [ ]:
def save_to_excel(merged_dfs):
    # Save as xlsx file
    merged_dfs.to_excel('pomiarPM25_lata_2014_2019_2024.xlsx', index=False)

In [ ]:
# Remove unnecessary rows
df2014_mod = df2014.drop([1, 2]).reset_index(drop=True)
df2019_mod = df2019.drop([0, 2, 3, 4, 5]).reset_index(drop=True)
df2024_mod = df2024.drop([0, 2, 3, 4, 5]).reset_index(drop=True)

# Process data, merge and save to file
all_years = [df2014_mod.copy(), df2019_mod.copy(), df2024_mod.copy()]
cleaned_dfs = clean_files(all_years)
save_to_excel(merge_dataframes(cleaned_dfs))

## Part 2

In [ ]:
# Reload the data
pm25_concentration = pd.read_excel("pomiarPM25_lata_2014_2019_2024.xlsx", index_col=0)

In [ ]:
# Monthly averages for each station and year
monthly_avg = pm25_concentration.copy()
monthly_avg.columns = [col.split('_')[-1] for col in monthly_avg.columns]

# Average / remove rows from irrelevant time periods / format index
monthly_avg = monthly_avg.resample('ME').mean()
monthly_avg = monthly_avg[monthly_avg.index.year.isin([2014, 2019, 2024])]
monthly_avg.index = monthly_avg.index.to_period('M')

display(monthly_avg)

In [ ]:
# Prepare values for the line chart for averaged values in Warsaw and Katowice
avg_warsaw_katowice = pm25_concentration.copy()
avg_warsaw_katowice.columns = [col.split('_')[0] for col in avg_warsaw_katowice.columns]

# Average / only data from 2014 and 2024 / format index / only Warsaw and Katowice
avg_warsaw_katowice = avg_warsaw_katowice.resample('ME').mean()
avg_warsaw_katowice = avg_warsaw_katowice[avg_warsaw_katowice.index.year.isin([2014, 2024])]
avg_warsaw_katowice.index = avg_warsaw_katowice.index.to_period('M')
avg_warsaw_katowice = avg_warsaw_katowice[['Warszawa', 'Katowice']]

# df with only Warsaw columns / remove Warsaw col from target df / add averaged Warsaw column
warsaw_columns = avg_warsaw_katowice.loc[:, avg_warsaw_katowice.columns == 'Warszawa']
avg_warsaw_katowice = avg_warsaw_katowice.drop(columns='Warszawa')
avg_warsaw_katowice["Warszawa"] = warsaw_columns.mean(axis=1)

In [ ]:
# Draw line charts
fig, axes = plt.subplots(1, 2, figsize=(20, 4), sharex=True, sharey=True)

years = [2014, 2024]
months = [i for i in range(1, 13)]

for (ax, year) in zip(axes, years):
    ax.plot(months, avg_warsaw_katowice['Warszawa'][avg_warsaw_katowice.index.year == year], 'o-', linewidth=2, markersize=5, color='red', label='Warsaw')
    ax.plot(months, avg_warsaw_katowice['Katowice'][avg_warsaw_katowice.index.year == year], 'o-', linewidth=2, markersize=5, color='blue', label='Katowice')
    ax.set_xlabel(f'Month/{year}', size=12)
    ax.set_ylabel(f'Average PM2.5 concentration', size=12)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(months)
    ax.legend()

fig.suptitle('Monthly average PM2.5 dust concentration in the air, in 2014 and 2024', size=15, weight='bold')
plt.show()

Description:
Dust concentration is highest in the first and last quarter of the year.
The air in Katowice is more polluted with PM2.5 than the air in Warsaw.

Differences:
Over 10 years, PM2.5 content in both cities has significantly decreased (especially in the first and last quarter).
The pollution difference between the cities has significantly decreased, although it is still observable in some months (November and December).

## Part 3 - Monthly averages heatmap

In [ ]:
def prepare_heatmap_data(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Processes data from the file to match the task requirements (groups by city, monthly averages).

    Args:
        df_input (pd.DataFrame): Merged input data from 2014, 2019, 2024 regarding PM2.5 pollution
    Returns:
        pd.DataFrame: Data in long format with columns [Year, Month, City, PM2.5], ready for visualization
    """
    df = df_input.copy()

    # Fix the first column - convert to dates, index by date
    df['date'] = pd.to_datetime(df['index'])
    df = df.set_index('date')
    df = df.drop(columns=['index'])

    # Extract city names from columns and group by time, taking the mean (same operations as in Part 2)
    df_monthly = df.resample('ME').mean()
    city_names = [col.split('_')[0] for col in df_monthly.columns]

    # For convenience: transpose, group by city names and compute means, then transpose back
    df_grouped_cities = df_monthly.T.groupby(city_names).mean().T

    # Prepare labels for long format
    df_grouped_cities['Year'] = df_grouped_cities.index.year
    df_grouped_cities['Month'] = df_grouped_cities.index.month

    # Prepare long format with separated columns for the chart
    df_long = df_grouped_cities.melt(id_vars=['Year', 'Month'], var_name='City', value_name='PM2.5')

    return df_long

In [ ]:
def create_heatmap(df_long: pd.DataFrame) -> None:
    """
    Draws a heatmap panel in pure Matplotlib based on data prepared by 'prepare_heatmap_data'.

    Args:
        df_long (pd.DataFrame): DataFrame with data formatted for creating the heatmap chart
    Returns:
        None: The function does not return a value, only displays the finished chart
    """
    # Get unique cities (because names in the table are duplicated) and sort alphabetically
    unique_cities = sorted(df_long['City'].unique())

    # Configure the chart grid - divide into smaller cells for subplots
    n_cols = 3
    n_rows = math.ceil(len(unique_cities) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), sharex=True, sharey=True)
    axes = axes.flatten()

    # Visual settings
    cmap = plt.get_cmap("RdYlGn_r")  # Red-Yellow-Green (reversed)
    vmin, vmax = 0, 60  # Color range
    years = [2014, 2019, 2024]

    # Draw the chart
    for i, city in enumerate(unique_cities):
        ax = axes[i]
        city_data = df_long[df_long['City'] == city]

        # Create a pivot matrix: rows = year, columns = month
        pivot_df = city_data.pivot(index='Year', columns='Month', values='PM2.5')

        # Force the table to have exactly 3 years and 12 months
        pivot_df = pivot_df.reindex(index=years, columns=range(1, 13))

        # Convert to numpy matrix (for imshow)
        data_matrix = pivot_df.to_numpy()

        # Draw (imshow)
        im = ax.imshow(data_matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect='auto')

        # Manually add numbers inside cells
        rows, cols = data_matrix.shape
        for r in range(rows):
            for c in range(cols):
                val = data_matrix[r, c]
                # Only print if the value exists (is not NaN)
                if not np.isnan(val):
                    ax.text(c, r, f"{val:.0f}", ha="center", va="center", color='white', fontsize=9)

        # Axis labels and title
        ax.set_title(city)
        ax.set_yticks(range(len(years)))
        ax.set_yticklabels(years)
        ax.set_ylabel('Year')
        ax.set_xticks(range(12))
        ax.set_xticklabels(range(1, 13))
        ax.set_xlabel('Month')

    # Adjust chart layout - leave a free margin on the right for the colorbar
    plt.tight_layout(rect=[0, 0, 0.9, 1])

    # Add the colorbar in that free space
    cbar_ax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
    fig.colorbar(im, cax=cbar_ax, label='Average PM2.5 concentration [µg/m3]')

    plt.suptitle('Monthly average PM2.5 concentrations in 2014, 2019 and 2024)', fontsize=20, y=1.02)
    plt.show()

In [ ]:
data = prepare_heatmap_data(merge_dataframes(cleaned_dfs))
create_heatmap(data)

Conclusion: In 2014, the cities with the highest PM2.5 pollution levels were Krakow, Legionowo and Katowice. We are now observing a trend of improving air quality in cities - even in large urban centres in 2024, pollution did not exceed 36 µg/m3.

## Part 4 - Days exceeding the WHO norm

In [ ]:
def count_exceedance_days(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Calculates the number of days in 2014, 2019 and 2024 on which the daily average PM2.5 concentration exceeded 15 µg/m3.

    Args:
        df_input (pd.DataFrame): DataFrame with hourly PM2.5 concentration data
    Returns:
        pd.DataFrame: Table indexed by year, containing the number of exceedance days for each station
    """
    df = df_input.copy()

    # Fix the first column - convert to dates, index by date
    df['date'] = pd.to_datetime(df['index'])
    df = df.set_index('date')
    df = df.drop(columns=['index'])

    # Calculate daily average
    df_daily = df.resample('D').mean()

    # If the norm is exceeded, mark it as 1. Then group by year and sum (equivalent to counting ones).
    mark_exceedance = df_daily > 15
    summed_exceedances = mark_exceedance.resample('YE').sum()

    # Change the index format from date to year only
    summed_exceedances.index = summed_exceedances.index.year

    # Select only rows for 2014, 2019, 2024
    years = [2014, 2019, 2024]
    years_present = [year for year in years if year in summed_exceedances.index]
    final_result = summed_exceedances.loc[years_present]

    return final_result

In [ ]:
def top3_exceedances(exceedance_summary: pd.DataFrame) -> (list[str], list[str]):
    """
    Identifies the 3 stations with the fewest and the 3 with the most total exceedance days.

    Args:
        exceedance_summary (pd.DataFrame): Table indexed by year, containing exceedance day counts per station
    Returns:
        tuple[list[str], list[str]]: Tuple containing two lists: the first with the 3 cleanest station names, the second with the most polluted
    """
    # Sort from smallest to largest total exceedance days per station
    sorted_results = exceedance_summary.sum().sort_values()

    best_results = sorted_results.head(3)  # cleanest
    worst_results = sorted_results.tail(3).iloc[::-1]  # most polluted

    best_list = [station for station in best_results.index]
    worst_list = [station for station in worst_results.index]

    return best_list, worst_list

In [ ]:
def create_grouped_barplot(df: pd.DataFrame) -> None:
    """
    Displays a grouped bar chart for the 3 stations with the fewest and 3 with the most days exceeding the WHO norm.

    Args:
        df (pd.DataFrame): DataFrame with hourly PM2.5 concentration data (input data for the calculations)
    Returns:
        None: The function does not return a value, only displays the finished chart
    """
    df_results = count_exceedance_days(df)
    best, worst = top3_exceedances(df_results)

    # Take data only for these six stations
    df_plot = df_results[best + worst].copy()

    # Configure axes and data
    years = [2014, 2019, 2024]
    stations = [col.split('_')[1] for col in df_plot.columns]
    x = np.arange(len(stations))  # positioning on the x axis
    width = 0.25
    offsets = [-width, 0, width]  # offsets for years - 2014 left, 2019 center, 2024 right
    colors = ['blue', 'red', 'green']

    # Draw the chart
    fig, ax = plt.subplots(figsize=(12, 8))

    for i, year in enumerate(years):
        # Draw bars and add numbers above them
        rects = ax.bar(x + offsets[i], df_plot.loc[year].values, width, label=str(year), color=colors[i], edgecolor='black')
        ax.bar_label(rects, fontsize=9)

    # Add axis labels, titles, dividing line between the two chart sections, and group annotations
    ax.set_ylabel('Number of exceedance days (>15 µg/m3)', fontsize=12)
    ax.set_title('Comparison of smog days at the 3 best and 3 worst stations', fontsize=14, pad=20)
    ax.set_xticks(x)
    ax.set_xticklabels(stations, fontsize=10)
    ax.axvline(x=2.5, color='gray', linestyle='--', linewidth=1)
    ax.text(1, ax.get_ylim()[1]*0.95, "Cleanest", ha='center', color='green')
    ax.text(4, ax.get_ylim()[1]*0.95, "Most polluted", ha='center', color='red')
    ax.legend(title='Year')

    plt.tight_layout()
    plt.show()

In [ ]:
create_grouped_barplot(merge_dataframes(cleaned_dfs))

Conclusion: The difference between the station recording the fewest days with PM2.5 pollution exceeding 15 µg/m3 and the station recording the most such days is on average around 150 days more. In 5 out of 6 of the presented stations, the average number of days exceeding WHO norms showed a downward trend over the presented time period.